# Similarity Search Benchmark: Large-Scale Symmetric Deletion Implementations

This notebook benchmarks implementations of the symmetric deletion algorithm at large dataset sizes (up to 30 million sequences): the pure-Python `pyrepseq.symdel`, `symscan`, and XTNeighbor (streaming, GPU-accelerated). Given a dataset of Adaptive Immune Receptor sequences and a Levenshtein distance threshold, each implementation identifies all pairs of sequences with a similarity below the threshold value. The dataset is obtained from [Emerson et al](https://doi.org/10.1038/ng.3822).

For a comparison across a broader set of algorithms at medium dataset sizes (up to 100,000 sequences), see the companion notebook `02_algorithms.ipynb`.

The notebook is divided into 3 steps as follows:
1. __Configuration:__ select the number of experiment repeats and maximum dataset size. Note that the largest option of dataset size requires a high-RAM VM which requires a paid Google Colab account.
2. __Benchmark Setup:__ install dependencies, compile XTNeighbor, and prepare input data.
3. __Symmetric Deletion Benchmark:__ perform benchmark on each implementation at threshold `d=1,2`, and across distances `d=1..5` at a fixed size. Requires a GPU runtime.

Warning: this notebook can take up to 1 hour to run.


## 1. Configuration

In [ ]:
# @title Configure the runtime and number of experiment repeats. High RAM is only available in Colab's premium plan.
n_repeat = 1 # @param ["1", "10", "30"] {type:"raw"}
high_ram = True # @param {type:"boolean"}

## 2. Benchmark Setup (run time < 5 min)

install dependency

In [ ]:
! pip install -q pyrepseq pybktree symscan

In [ ]:
import os.path
import numpy as np
import pandas as pd
import seaborn as sns
import time
import random
import pybktree
from pyrepseq import levenshtein_distance
import pyrepseq
import symscan

import sys

import benchutils
from benchutils import BenchmarkTimeout, run_binary, time_limit, describe_env

try:
    from google.colab import files
    colab = True
except ImportError:
    colab = False

if colab and not os.path.exists("XT-neighbor"):
    !git clone https://github.com/heartnetkung/XT-neighbor.git
    repo_path = "XT-neighbor/"
else:
    repo_path = "../"

In [ ]:
benchutils.timeout_seconds = 100

In [ ]:
describe_env()

compile XTNeighbor without streaming

In [ ]:
! mkdir -p {repo_path}/xtneighbor/build
! cd {repo_path}/xtneighbor/build; cmake -S .. -B .;make

compile XTNeighbor

In [ ]:
! mkdir -p {repo_path}/xtneighbor_streaming/build
! cd {repo_path}/xtneighbor_streaming/build; cmake -S .. -B .;make

prepare input data

In [ ]:
N_FILES=6

data = []
for i in range(1,N_FILES+1):
  data += pd.read_csv(f'{repo_path}/data/emerson{i}.zip', compression='zip', header=0)['cdr3'].to_list()

print('first row:', data[0]);
print(f'len: {len(data):,}')

In [ ]:
# garbage collect following large data import
# freeze imported data to avoid reconsideration during benchmarking
import gc
gc.collect()
gc.freeze()

## 3. Symmetric Deletion  Implementation Benchmark (run time ~ 10 min at n_repeat=1)

check GPU availability

In [ ]:
import subprocess
try:
  subprocess.run(["nvidia-smi"], capture_output=True, text=True)
except Exception as e:
  raise Exception("GPU required")

symscan measurement worker (run out-of-process, see run_symscan below)

In [ ]:
! mkdir -p tmp

In [ ]:
%%writefile tmp/symscan_worker.py
# Standalone worker run as a fresh subprocess for each symscan measurement, so that a
# benchmark timeout can SIGKILL the whole process group instead of unwinding this
# interpreter. 
# Prints the seconds spent inside get_neighbors_within; interpreter startup, input
# reading and one-time library initialization are excluded, so the number is
# comparable to the in-process wall-clock timings of the other CPU algorithms.
import gc
import sys
import time

import symscan

# warmup the same as in-process
WARMUP_N = 100

def main():
    seq_file, max_distance = sys.argv[1], int(sys.argv[2])

    with open(seq_file) as f:
        seqs = [line.strip() for line in f if line.strip()]

    symscan.get_neighbors_within(seqs[:WARMUP_N], max_distance=max_distance)
    gc.collect()
    gc.freeze()

    t0 = time.perf_counter()
    symscan.get_neighbors_within(seqs, max_distance=max_distance)
    print(time.perf_counter() - t0)


if __name__ == '__main__':
    main()


standardize all algorithms to the same API

In [ ]:
def prepare(seqs):
  with open("input1.txt","w") as file1:
    file1.writelines(seq+'\n' for seq in seqs)
  with open("input2.txt","w") as file2:
    file2.writelines(seq+'\n' for seq in (['cdr3']+seqs))

def xt_neighbor_1(seqs,threshold,_len,verbose=False): #verbose is ignored
  run_binary([f'{repo_path}/xtneighbor/build/xt_neighbor',
              '-p', 'input1.txt', '-n', str(_len), '-d', str(threshold)])

def xt_neighbor_2(seqs,threshold,_len,verbose=False):
  cmd = [f'{repo_path}/xtneighbor_streaming/build/xt_neighbor',
         '-i', 'input2.txt', '-n', str(_len), '-d', str(threshold)]
  if verbose:
    cmd.append('-V')
  run_binary(cmd)

def symdel(seqs,threshold,_len,verbose=False): #verbose is ignored
  return pyrepseq.symdel(seqs,max_edits=threshold)

SYMSCAN_WORKER = 'tmp/symscan_worker.py'

def run_symscan(seqs,threshold,_len,verbose=False):
  # measured inside the worker, so interpreter startup and reading input1.txt are
  # excluded; returns that runtime instead of letting the loop time the subprocess
  out = run_binary([sys.executable, SYMSCAN_WORKER, 'input1.txt', str(threshold)],
                   echo=verbose)
  try:
    return float(out.strip().splitlines()[-1])
  except (ValueError, IndexError):
    raise RuntimeError(f'{SYMSCAN_WORKER} produced no runtime:\n{out}') from None



benchmarking code

In [ ]:
# 100 is the warm up
sizes = [100, 10_000, 30_000, 100_000, 300_000, 1_000_000, 3_000_000, 10_000_000, 30_000_000]

limits = {}
#    'symscan_1':30_000_000,
#    'symscan_2':10_000_000,
#    'V1_1':10_000_000,
#    'V1_2':1_000_000,
#    'symdel_1':3_000_000,
#    'symdel_2':1_000_000,
#    'V2_1':30_000_000,
#    'V2_2':10_000_000}
limits.update({f"V1_{dist}": 0 for dist in [3,4,5]})

algorithms = {
    'V2':xt_neighbor_2,
    'symdel':symdel,
    'symscan':run_symscan,
}

# algorithms that measure themselves (out-of-process) and return their own runtime
# instead of being timed by the loop's wall clock
self_timed = {'symscan'}
verbose = False

symdel_result = {'runtime':[],'algorithm':[],'input_size':[],'distance':[]}

def run_exp(distances, sizes=sizes):
    for i in range(n_repeat):
        for size in sizes:
            subset = random.Random(i).sample(data,size)
            prepare(subset)
            for distance in distances:
                for alg_name in algorithms:
                    limit = limits.get(f"{alg_name}_{distance}")
                    if limit is not None and limit < size:
                        print(f"Skipping {alg_name} for size {size:,} and distance {distance} due to limit {limit}")
                        continue

                    # perform
                    start = time.time()
                    try:
                        with time_limit():
                            measured = algorithms[alg_name](subset,distance,size,verbose)
                    except BenchmarkTimeout:
                        print(f'Timeout: {alg_name} for size {size:,} and distance {distance} exceeded {benchutils.timeout_seconds}s')
                        limits[f"{alg_name}_{distance}"] = size - 1
                        continue
                    end = time.time()
                    runtime = measured if alg_name in self_timed else end - start

                    # record
                    print(f'{size:,}', alg_name, distance, i, round(runtime*100)/100)
                    symdel_result['runtime'].append(runtime)
                    symdel_result['algorithm'].append(alg_name)
                    symdel_result['input_size'].append(size)
                    symdel_result['distance'].append(distance)

In [ ]:
run_exp(distances=[1])

In [ ]:
run_exp(distances=[2])

In [ ]:
pd.DataFrame(symdel_result).to_csv('../data/gpu_benchmark.csv')
if colab:
    files.download('gpu_benchmark.csv')

In [ ]:
symdel_result = {'runtime':[],'algorithm':[],'input_size':[],'distance':[]}

run_exp(distances=range(1,6), sizes=[100_000])

In [ ]:
pd.DataFrame(symdel_result).to_csv('../data/gpu_dist_benchmark.csv')
if colab:
    files.download('gpu_dist_benchmark.csv')